In [1]:
import itertools
import os
import subprocess
import glob, os, shutil, sys
import numpy as np
import pandas as pd
import xarray as xr
import time

In [ ]:
def make_namelist(station_list,subgrid,VGCB,VCF,RUNOFF,RSS,SCW,PHS,WUE,SP_SO_SN,path,model_dir,case_number):
    nml_file_path = "/home/liuz21/stu01/github/single_test/US-Wkg_IGBP_CB.nml"  # model nml file
    # open nml file and readlines
    with open(nml_file_path, 'r') as file:
        nml_content = file.readlines()

    # modify var
    for i, line in enumerate(nml_content):
        # replace CASE NAME
        if 'DEF_CASE_NAME' in line:
            nml_content[i] = f"DEF_CASE_NAME = '{subgrid}-{VGCB}-{case_number}'\n"
        
        # replace SITE DATA
        if 'USE_SITE_pctpfts' in line:
            if subgrid == 'igbp' or subgrid == 'usgs':
                nml_content[i] = f"USE_SITE_pctpfts = .false.\n"
            elif subgrid == 'pft' or subgrid == 'pc':
                nml_content[i] = f"USE_SITE_pctpfts = .true.\n"
        if 'SITE_fsitedata' in line:
            nml_content[i] = f"SITE_fsitedata = '{station_list['srfdata_dir']}{station_list['srfdata_name']}'\n"
        # replace SITE SIMULATION 
        if 'DEF_simulation_time%start_year' in line:
            nml_content[i] = f"DEF_simulation_time%start_year = {station_list['Syear']}\n"
        if 'DEF_simulation_time%end_year' in line:
            nml_content[i] = f"DEF_simulation_time%end_year = {station_list['Eyear']}\n"
        if 'DEF_simulation_time%spinup_year' in line:
            nml_content[i] = f"DEF_simulation_time%spinup_year = {int(station_list['Syear']) - 1}\n"
        #  replace PATH
        if 'DEF_dir_output' in line:
            nml_content[i] = f"DEF_dir_output = '{model_dir}{station_list['SiteName']}/'\n"
        if 'DEF_forcing_namelist' in line:
            nml_content[i] = f"DEF_forcing_namelist = '{station_list['metdata_dir']}{station_list['SiteName']}.nml'\n"
        # replace SCHEME
        if 'DEF_USE_VariablySaturatedFlow' in line:
            nml_content[i] = f"DEF_USE_VariablySaturatedFlow = .{VCF}. \n"
        if 'DEF_Runoff_SCHEME' in line:
            nml_content[i] = f"DEF_Runoff_SCHEME = {RUNOFF} \n"
        if 'DEF_RSS_SCHEME' in line:
            nml_content[i] = f"DEF_RSS_SCHEME = {RSS} \n"
        if 'DEF_USE_SUPERCOOL_WATER' in line:
            nml_content[i] = f"DEF_USE_SUPERCOOL_WATER = .{SCW}. \n"
        if 'DEF_USE_PLANTHYDRAULICS' in line:
            nml_content[i] = f"DEF_USE_PLANTHYDRAULICS = .{PHS}. \n"
        if 'DEF_USE_WUEST' in line:
            nml_content[i] = f"DEF_USE_WUEST = .{WUE}. \n"
        if 'DEF_SPLIT_SOILSNOW' in line:
            nml_content[i] = f"DEF_SPLIT_SOILSNOW = .{SP_SO_SN}. \n"    
    # read back modified nml file
    # sometimes need modify nml file path
    pathdir = f"{path}{subgrid}-{VGCB}/nml/{case_number}/"
    isExist = os.path.exists(pathdir)
    if not isExist:
        os.makedirs(pathdir)
    new_file_path = f"{pathdir}{station_list['SiteName']}.nml"
    with open(new_file_path, 'w') as file:
        file.writelines(nml_content)

def is_valid_combination(param_dict):
    if param_dict['VGCB'] == 'cb' and param_dict['RUNOFF'] == '3':
        return False
    return True


if __name__ == '__main__':
    
    
    subgrid = ["usgs","igbp", "pft", "pc"]  
    VGCB = ["cb", "vg"]              
    params_df = pd.read_csv('params.csv')

    path = '/share/home/dq076/mode/ME/CoLM202X_CH4_s/single_point/'
    base_output_path = '/share/home/dq076/mode/ME/CoLM202X_CH4_s/single_point_test/'
    #stnlist = f"/home/liuz21/stu01/github/CoLM202X/run/LIST_Land_selected.csv"
    stnlist = f"LIST_Land_select_rest.csv"
    station_lists = pd.read_csv(stnlist, header=0)  #flux Sheet2,header=0
    n = len(station_lists['SiteName'])
    number = ["2","5","8"]
    for grid in range(2,4):
        for vgcb_n in range(0,1):
            sub     = subgrid[grid]
            hydro   = VGCB[vgcb_n]
            print(f"Subgrid: {sub}, Hydro: {hydro}")
            for i in range(0,3):
                case_number = number[i]
                #case_number = num
                param_dict = params_df.loc[int(case_number) - 1]
                VCF     = param_dict['VSF']
                RUNOFF  = param_dict['RUNOFF']
                RSS     = param_dict['RSS']
                SCW     = param_dict['SCW']
                PHS     = param_dict['PHS']
                WUE     = param_dict['WUE']
                SP_SO_SN    = param_dict['SP_SO_SN']
                output_dir = os.path.join(path, f"{sub}-{hydro}/nml/{case_number}")
                # 创建目录
                os.makedirs(output_dir, exist_ok=True)
                # 输出到每个目录（这里以写入一个配置文件为示例）
                config_file_path = os.path.join(output_dir, "config.txt")
                with open(config_file_path, "w") as f:
                    for key, value in param_dict.items():
                        f.write(f"{key} = {value}\n")
                for i in range(n):
                    station_list = station_lists.iloc[i]        
                    make_namelist(station_list,sub,hydro,VCF,RUNOFF,RSS,SCW,PHS,WUE,SP_SO_SN,path,base_output_path, case_number)